In [1]:
!pip install -q mlflow

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error
import mlflow
from mlflow import MlflowClient

print("MLflow version:", mlflow.__version__)

MLflow version: 3.18.0


In [2]:
np.random.seed(42)
number_of_orders = 500

df = pd.DataFrame()
df["distance_km"] = np.random.uniform(1, 15, number_of_orders)
df["items"]       = np.random.randint(1, 10, number_of_orders)
df["hour"]        = np.random.randint(8, 23, number_of_orders)

noise = np.random.normal(0, 3, number_of_orders)
df["delivery_minutes"] = 10 + 3 * df["distance_km"] + 1.5 * df["items"] + noise

X = df[["distance_km", "items", "hour"]]
y = df["delivery_minutes"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print("Training rows:", len(X_train))
print("Test rows    :", len(X_test))
df.head()

Training rows: 400
Test rows    : 100


,distance_km,items,hour,delivery_minutes
0,6.243562,6,18,36.064045
1,14.310000,1,21,55.905380
2,11.247915,9,14,58.715004
3,9.381219,1,8,37.832316
4,3.184261,5,9,26.350811


In [3]:
for depth in [2, 5, 10]:
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    mae = mean_absolute_error(y_test, predictions)
    print("max_depth =", depth, "   MAE =", round(mae, 2))

max_depth = 2    MAE = 4.59
max_depth = 5    MAE = 3.31
max_depth = 10    MAE = 4.07


In [4]:
mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("P06-delivery")
print("Tracking URI:", mlflow.get_tracking_uri())

with mlflow.start_run(run_name="first-tree"):
    depth = 4
    model = DecisionTreeRegressor(max_depth=depth, random_state=42)
    model.fit(X_train, y_train)
    mae = mean_absolute_error(y_test, model.predict(X_test))

    mlflow.log_param("max_depth", depth)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_metric("mae_minutes", mae)
    mlflow.set_tag("student", "demo")

print("Run recorded. MAE =", round(mae, 2))   # expect 3.47

2026/11/14 19:45:36 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/11/14 19:45:36 INFO mlflow.store.db.utils: Updating database tables
2026/11/14 19:45:39 INFO mlflow.tracking.fluent: Experiment with name 'P06-delivery' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow.db
Run recorded. MAE = 3.47


In [5]:
runs = mlflow.search_runs(experiment_names=["P06-delivery"])
display(runs[["tags.mlflow.runName", "params.max_depth", "metrics.mae_minutes"]])

depth_values = [1, 2, 3, 5, 8, 12]
for depth in depth_values:
    with mlflow.start_run(run_name="tree-depth-" + str(depth)):
        model = DecisionTreeRegressor(max_depth=depth, random_state=42)
        model.fit(X_train, y_train)
        mae = mean_absolute_error(y_test, model.predict(X_test))
        mlflow.log_param("max_depth", depth)
        mlflow.log_metric("mae_minutes", mae)
    print("max_depth =", depth, "   MAE =", round(mae, 2))

,tags.mlflow.runName,params.max_depth,metrics.mae_minutes
0,first-tree,4,3.465814


max_depth = 1    MAE = 6.11
max_depth = 2    MAE = 4.59
max_depth = 3    MAE = 4.04
max_depth = 5    MAE = 3.31
max_depth = 8    MAE = 3.96
max_depth = 12    MAE = 4.39


In [6]:
best = mlflow.search_runs(
    experiment_names=["P06-delivery"],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.mae_minutes ASC"],
    max_results=1,
)

best_run_name = best["tags.mlflow.runName"][0]
best_depth    = best["params.max_depth"][0]
best_mae      = best["metrics.mae_minutes"][0]
print("Best run  :", best_run_name)
print("Best depth:", best_depth)
print("Best MAE  :", round(best_mae, 2))

best_depth = int(best_depth)   # params are stored as text

with mlflow.start_run(run_name="winner-saved"):
    model = DecisionTreeRegressor(max_depth=best_depth, random_state=42)
    model.fit(X_train, y_train)
    mae = mean_absolute_error(y_test, model.predict(X_test))

    mlflow.log_param("max_depth", best_depth)
    mlflow.log_metric("mae_minutes", mae)

    mlflow.sklearn.log_model(
        sk_model=model,
        name="model",
        registered_model_name="delivery-time-model",
        serialization_format="cloudpickle",
    )

print("Model saved and registered. MAE =", round(mae, 2))

2026/11/14 19:46:49 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Best run  : tree-depth-5
Best depth: 5
Best MAE  : 3.31
Model saved and registered. MAE = 3.31


Successfully registered model 'delivery-time-model'.
Created version '1' of model 'delivery-time-model'.


In [7]:
client = MlflowClient()
versions = client.search_model_versions("name='delivery-time-model'")
for v in versions:
    print("Version:", v.version, "   Created by run:", v.run_id)

loaded_model = mlflow.sklearn.load_model("models:/delivery-time-model/1")

new_order = pd.DataFrame({"distance_km": [8], "items": [3], "hour": [19]})
predicted_time = loaded_model.predict(new_order)[0]
print("Predicted delivery time:", round(predicted_time, 1), "minutes")   # about 42.4

Version: 1    Created by run: 85f8c8f9c5314426917ba9aaf9f4a809
Predicted delivery time: 42.4 minutes


In [8]:
import subprocess, sys, time
from google.colab import output

ui_process = subprocess.Popen([
    sys.executable, "-m", "mlflow", "ui",
    "--backend-store-uri", "sqlite:///mlflow.db",
    "--port", "5001",
    "--allowed-hosts", "*",
])

time.sleep(15)
output.serve_kernel_port_as_window(5001)   # click the link that appears

Try `serve_kernel_port_as_iframe` instead. 


<IPython.core.display.Javascript object>

In [10]:
from sklearn.datasets import load_breast_cancer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score

data = load_breast_cancer(as_frame=True)
X, y = data.data, data.target
print(data.target_names, y.value_counts().to_dict())   # 0 = malignant, 1 = benign

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

mlflow.set_tracking_uri("sqlite:///mlflow.db")
mlflow.set_experiment("cancer-diagnosis")

def evaluate(model):
    preds = model.predict(X_test)
    return accuracy_score(y_test, preds), f1_score(y_test, preds)

['malignant' 'benign'] {1: 357, 0: 212}


In [11]:
with mlflow.start_run(run_name="logreg-baseline"):
    model = LogisticRegression(max_iter=10000, random_state=42)
    model.fit(X_train, y_train)
    acc, f1 = evaluate(model)

    mlflow.log_param("model_type", "LogisticRegression")
    mlflow.log_param("max_iter", 10000)
    mlflow.log_param("test_size", 0.2)
    mlflow.log_metric("accuracy", acc)
    mlflow.log_metric("f1_score", f1)
    mlflow.set_tag("sweep", "baseline")
    mlflow.sklearn.log_model(sk_model=model, name="model", serialization_format="cloudpickle")

print("Baseline  accuracy =", round(acc, 4), "  F1 =", round(f1, 4))

2026/11/14 19:49:09 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Baseline  accuracy = 0.9649   F1 = 0.9726


In [12]:
for n_est in [10, 50, 100]:
    for depth in [3, 5, 10]:
        with mlflow.start_run(run_name=f"rf-n{n_est}-d{depth}"):
            model = RandomForestClassifier(n_estimators=n_est, max_depth=depth, random_state=42)
            model.fit(X_train, y_train)
            acc, f1 = evaluate(model)

            mlflow.log_param("model_type", "RandomForest")
            mlflow.log_param("n_estimators", n_est)
            mlflow.log_param("max_depth", depth)
            mlflow.log_param("test_size", 0.2)
            mlflow.log_metric("accuracy", acc)
            mlflow.log_metric("f1_score", f1)
            mlflow.set_tag("sweep", "rf")
            mlflow.sklearn.log_model(sk_model=model, name="model", serialization_format="cloudpickle")

        print(f"n_estimators={n_est:<3} max_depth={depth:<2} accuracy={acc:.4f} F1={f1:.4f}")

2026/11/14 19:49:20 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/11/14 19:49:26 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


n_estimators=10  max_depth=3  accuracy=0.9386 F1=0.9510


2026/11/14 19:49:33 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


n_estimators=10  max_depth=5  accuracy=0.9561 F1=0.9655
n_estimators=10  max_depth=10 accuracy=0.9386 F1=0.9510


2026/11/14 19:49:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


n_estimators=50  max_depth=3  accuracy=0.9474 F1=0.9583


... (6 outputs left out)

2026/11/14 19:49:54 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


n_estimators=100 max_depth=5  accuracy=0.9561 F1=0.9655


2026/11/14 19:49:58 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


n_estimators=100 max_depth=10 accuracy=0.9561 F1=0.9655


In [13]:
runs = mlflow.search_runs(
    experiment_names=["cancer-diagnosis"],
    filter_string="attributes.status = 'FINISHED'",
    order_by=["metrics.f1_score DESC", "metrics.accuracy DESC"],  # accuracy breaks F1 ties
)

table = runs[["tags.mlflow.runName", "tags.sweep", "params.n_estimators",
              "params.max_depth", "metrics.accuracy", "metrics.f1_score"]]
display(table)    # screenshot this table

best_run_id   = runs["run_id"][0]
best_run_name = runs["tags.mlflow.runName"][0]
print("Best run by F1:", best_run_name, "| F1 =", round(runs["metrics.f1_score"][0], 4))

,tags.mlflow.runName,tags.sweep,params.n_estimators,params.max_depth,metrics.accuracy,metrics.f1_score
0,logreg-baseline,baseline,None,None,0.964912,0.972603
1,rf-n100-d10,rf,100,10,0.956140,0.965517
2,rf-n100-d5,rf,100,5,0.956140,0.965517
3,rf-n100-d3,rf,100,3,0.956140,0.965517
4,rf-n50-d10,rf,50,10,0.956140,0.965517
5,rf-n50-d5,rf,50,5,0.956140,0.965517
6,rf-n10-d5,rf,10,5,0.956140,0.965517
7,rf-n50-d3,rf,50,3,0.947368,0.958333
8,rf-n10-d10,rf,10,10,0.938596,0.951049
9,rf-n10-d3,rf,10,3,0.938596,0.951049


Best run by F1: logreg-baseline | F1 = 0.9726


In [14]:
result = mlflow.register_model(f"runs:/{best_run_id}/model", "cancer-classifier")
print("Registered version:", result.version)

loaded = mlflow.sklearn.load_model("models:/cancer-classifier/1")

first_five = X_test.iloc[:5]
print("Predicted:", loaded.predict(first_five))
print("Actual   :", y_test.iloc[:5].values)

Successfully registered model 'cancer-classifier'.
2026/11/14 19:50:22 WARNING mlflow.tracking._model_registry.fluent: Run with id 7ebb739018c240a486ba9b802586203e has no artifacts at artifact path 'model', registering model based on models:/m-43e245e501fa41daad608e80cb21218f instead
Created version '1' of model 'cancer-classifier'.


Registered version: 1
Predicted: [0 1 0 1 0]
Actual   : [0 1 0 1 0]
